In [1]:
import copy
import os
import torch
from utils.utils_poses.align_traj import align_ate_c2b_use_a2b
import scipy
from utils.utils_poses.comp_ate import compute_rpe, compute_ATE
import numpy as np
from utils.vis_utils import plot_pose
from colmap_parsing_utils import read_images_binary
from pathlib import Path
from nerfstudio.utils.io import load_from_json
from utils.utils_poses.lie_group_helper import convert3x4_4x4
from colmap_parsing_utils import read_images_binary,read_images_text
from nerfstudio.data.utils import colmap_parsing_utils as colmap_utils

# Setting EXPNAME



In [2]:
exp_name="base+abs+cull+antialiased+split25k+inter"

In [3]:
def align_pose(pose1, pose2):
    mtx1 = np.array(pose1, dtype=np.double, copy=True)
    mtx2 = np.array(pose2, dtype=np.double, copy=True)

    if mtx1.ndim != 2 or mtx2.ndim != 2:
        raise ValueError("Input matrices must be two-dimensional")
    if mtx1.shape != mtx2.shape:
        raise ValueError("Input matrices must be of same shape")
    if mtx1.size == 0:
        raise ValueError("Input matrices must be >0 rows and >0 cols")

    # translate all the data to the origin
    mtx1 -= np.mean(mtx1, 0)
    mtx2 -= np.mean(mtx2, 0)

    norm1 = np.linalg.norm(mtx1)
    norm2 = np.linalg.norm(mtx2)

    if norm1 == 0 or norm2 == 0:
        raise ValueError("Input matrices must contain >1 unique points")

    # change scaling of data (in rows) such that trace(mtx*mtx') = 1
    mtx1 /= norm1
    mtx2 /= norm2

    # transform mtx2 to minimize disparity
    R, s = scipy.linalg.orthogonal_procrustes(mtx1, mtx2)
    mtx2 = mtx2 * s

    return mtx1, mtx2, R

def eval_pose(poses_gt_c2w,poses_pred_c2w,output_path):
    poses_gt = poses_gt_c2w[:len(poses_pred_c2w)].clone()
    # align scale first (we do this because scale differennt a lot)
    trans_gt_align, trans_est_align, _ = align_pose(poses_gt[:, :3, -1].numpy(),
                                                            poses_pred_c2w[:, :3, -1].numpy())
    poses_gt[:, :3, -1] = torch.from_numpy(trans_gt_align)
    poses_pred_c2w[:, :3, -1] = torch.from_numpy(trans_est_align)

    c2ws_est_aligned = align_ate_c2b_use_a2b(poses_pred_c2w, poses_gt)
    ate = compute_ATE(poses_gt.cpu().numpy(),
                        c2ws_est_aligned.cpu().numpy())
    rpe_trans, rpe_rot = compute_rpe(
        poses_gt.cpu().numpy(), c2ws_est_aligned.cpu().numpy())
    print("RPE_t : {0:.3f}".format(rpe_trans*100),
            '&' "RPE_r : {0:.3f}".format(rpe_rot * 180 / np.pi),
            '&', "ATE : {0:.3f}".format(ate))
    pose_vis_path=plot_pose(poses_gt.cpu().numpy(), c2ws_est_aligned.cpu().numpy(), output_path)
    # # pdb.set_trace()
    with open(f"{output_path}/pose_eval.txt", 'w') as f:
        f.write("RPE_trans: {:.03f}, RPE_rot: {:.03f}, ATE: {:.03f}".format(
            rpe_trans*100,
            rpe_rot * 180 / np.pi,
            ate))
        f.close()



In [4]:
def load_c2w_pose(pose_path =None ,pose_type='colmap'):
    assert pose_path is not None, "pose_path should be provided"
    
    data_path=Path(pose_path)
    if pose_type=='colmap':
        images_info=read_images_text(data_path / "images.txt")
        filename_pose_dict = {}
        image_filenames = []
        poses = []
        for info in images_info.values():
            image_filenames.append(info.name)
            rotation = colmap_utils.qvec2rotmat(info.qvec)
            translation = info.tvec.reshape(3, 1)
            w2c = np.concatenate([rotation, translation], 1)
            w2c = np.concatenate([w2c, np.array([[0, 0, 0, 1]])], 0)
            c2w=np.eye(4)
            # c2w = np.linalg.inv(w2c)
            c2w[:3,:3]=rotation.T
            c2w[:3,3]=np.dot(-rotation.T,translation).squeeze()
            # Convert from COLMAP's camera coordinate system (OpenCV) to ours (OpenGL)
            c2w[0:3, 1:3] *= -1
            poses.append(c2w)
    elif pose_type=='ns-export':
        split="train"
        meta = load_from_json(data_path / f"transforms_{split}.json")
        image_filenames = []
        poses = []
        filename_pose_dict = {}
        for frame in meta:
            fname = frame["file_path"].replace("./", "").split('/')[-1]
            image_filenames.append(fname)
            poses.append(convert3x4_4x4(np.array(frame["transform"])))
            
        # poses = np.array(poses).astype(np.float32)
        # pred_pose_c2w = torch.from_numpy(poses[:, :3])  # camera to world transform
        # pred_pose_c2w=convert3x4_4x4(pred_pose_c2w)
    else:
        raise ValueError("pose_type should be colmap or export")
        
    filename_pose_dict=dict(zip(image_filenames,poses))
    return filename_pose_dict


In [5]:

for scene in sorted(["Horse", "Museum"  , "Church" ,"Family"  , "Ignatius" , "Ballroom", "Barn"]):
# get camera pose
    print(f"scene:{scene}")
    key_pose=f"outputs/{exp_name}/{scene}"
    filename_pred_pose_dict=load_c2w_pose(pose_path =key_pose ,pose_type='ns-export')

    gt_pose=f"dataset/Tanks/{scene}/sparse/0"
    filename_gt_pose_dict=load_c2w_pose(pose_path =gt_pose ,pose_type='colmap')

    # get the sorted pose
    pred_c2w_order=sorted(filename_pred_pose_dict.items(), key=lambda x:x[0])
    pred_c2w_list=[]
    gt_c2w_list=[]
    for i in range(len(pred_c2w_order)):
        file_name=pred_c2w_order[i][0]
        pred_pose=filename_pred_pose_dict[file_name]
        
        pred_c2w_list.append(pred_pose)
        gt_pose=filename_gt_pose_dict[file_name]
        gt_c2w_list.append(gt_pose)
    pred_c2w=torch.from_numpy(np.stack(pred_c2w_list,axis=0))
    gt_c2w=torch.from_numpy(np.stack(gt_c2w_list,axis=0))

    #evaluate the pose

    data_path=f"outputs/{exp_name}/{scene}/"
    # print(data_path)
    eval_pose(gt_c2w,pred_c2w,data_path)
        

scene:Ballroom
RPE_t : 0.016 &RPE_r : 0.014 & ATE : 0.000
scene:Barn
RPE_t : 0.008 &RPE_r : 0.016 & ATE : 0.001
scene:Church
RPE_t : 0.007 &RPE_r : 0.013 & ATE : 0.000
scene:Family
RPE_t : 0.012 &RPE_r : 0.012 & ATE : 0.000
scene:Francis


FileNotFoundError: [Errno 2] No such file or directory: 'outputs/base+abs+cull+antialiased+split25k+inter/Francis/transforms_train.json'